# Match labelled rows with LLM-extracted rows
1. Load data
2. Vectorize rows
3. Match rows 
4. Compute accuracy


In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import geopy as gpy
import time
import itertools
import regex as re
from matplotlib import pyplot as plt
from src.data import *
from src.plot_functions import *
from src.post_process_functions import *
from src.geocoding import *
from src.hazard_def import *
from src.impact_def import *
from src.sanity_checks import *



## Load data

In [2]:
#load data (model)
post_processed = True #load or not post_processed data
model_name = "meta-llama/llama-4-scout-17b-16e-instruct"
nreports = 50
res_savename = f"post_processed_llm_response_impact_labelled_reports_{model_name.replace('/', '_')}.csv" if post_processed else f"llm_response_impact_labelled_reports_{model_name.replace('/', '_')}.csv"
extracted_df = pd.read_csv(DATA_OUT_LLMS+res_savename)

#load data (labelled)
res_savename = "post_processed_labelled_reports_impacts_all.csv" if post_processed else "labelled_reports_impacts_all.csv"
labelled_df = pd.read_csv(DATA_OUT_LLMS+res_savename)


In [3]:
#reformat output
num_cols = ["impactValue"]#"startYear", "startMonth", "startDay", "endYear", "endMonth", "endDay"
list_cols = ["country","location", "hazards", "impactsAnnotation"]
labelled_df = format_output(labelled_df, num_cols=num_cols, list_cols=list_cols)
extracted_df = format_output(extracted_df, num_cols=num_cols, list_cols=list_cols)
#labelled_df.replace(np.nan, None, inplace=True)
#extracted_df.replace(np.nan, None, inplace=True)


In [4]:
extracted_df

,impactSubtype,impactValue,impactUnit,impactValuePrecision,country,location,startYear,startMonth,startDay,endYear,...,reportLink,disasterType,nathaz_text,impactType,country_iso3,country_iso3_kw,hazards_reclass,impactValueOrig,impactUnitOrig,unit_type
0,Affected People,43880.0,people,exact,"[Kiribati, Papua New Guinea, Solomon Islands, ...",[],NaN,NaN,NaN,NaN,...,https://adore.ifrc.org/Download.aspx?FileId=16...,Cyclone,['He explained clearly what the indicators mea...,NaN,NaN,FJI,['Unknown'],43880.0,people,other
1,Affected People,34573.0,people,exact,"[Kiribati, Papua New Guinea, Solomon Islands, ...",[],NaN,NaN,NaN,NaN,...,https://adore.ifrc.org/Download.aspx?FileId=16...,Cyclone,['He explained clearly what the indicators mea...,NaN,NaN,FJI,['Unknown'],34573.0,people,other
2,Residential Buildings,900.0,homes,exact,[Vanuatu],[West Tanna],NaN,NaN,NaN,NaN,...,https://adore.ifrc.org/Download.aspx?FileId=16...,Cyclone,['He explained clearly what the indicators mea...,NaN,NaN,FJI,['Unknown'],900.0,houses,other
3,Human Health and Wellbeing,85.0,unknown,exact,[Vanuatu],[],NaN,NaN,NaN,NaN,...,https://adore.ifrc.org/Download.aspx?FileId=16...,Cyclone,['He explained clearly what the indicators mea...,NaN,NaN,FJI,['Unknown'],85.0,per cent,other
4,"Access to Water, Sanitation, and Hygiene",414.0,people,exact,[Tuvalu],[],NaN,NaN,NaN,NaN,...,https://adore.ifrc.org/Download.aspx?FileId=16...,Cyclone,['He explained clearly what the indicators mea...,NaN,NaN,FJI,['Unknown'],138.0,households,other
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
270,Crop Production and Forestry,NaN,undefined crop production and forestry,NaN,[Zambia],[],2023.0,NaN,NaN,2024.0,...,https://adore.ifrc.org/Download.aspx?FileId=84...,Drought,['SITUATION ANALYSIS Description of the crisis...,NaN,NaN,ZMB,['Drought'],NaN,NaN,other
271,Affected Livestock and Animals,NaN,undefined affected animals,NaN,[Zambia],[],2023.0,NaN,NaN,2024.0,...,https://adore.ifrc.org/Download.aspx?FileId=84...,Drought,['SITUATION ANALYSIS Description of the crisis...,NaN,NaN,ZMB,['Drought'],NaN,NaN,other
272,Other Economic and Livelihood Impacts,11.0,CHF,exact,[Zambia],[],2024.0,NaN,NaN,NaN,...,https://adore.ifrc.org/Download.aspx?FileId=84...,Drought,['SITUATION ANALYSIS Description of the crisis...,NaN,NaN,ZMB,['Drought'],11.0,CHF million,other
273,Access to Food,NaN,people,NaN,[Zambia],[],NaN,NaN,NaN,NaN,...,https://adore.ifrc.org/Download.aspx?FileId=84...,Drought,['SITUATION ANALYSIS Description of the crisis...,NaN,NaN,ZMB,['Drought'],NaN,NaN,other


## Vectorize


In [49]:
def vectorize(cell_values, unique_values):
    """vectorizing function for categorical columns"""
    #cell_values = list() if not cell_values else cell_values
    cell_values = [cell_values] if not isinstance(cell_values, list) else cell_values
    vector = [1 if unique_value in cell_values else 0 for unique_value in unique_values]
    return np.array(vector)#.reshape(1, -1)

unique_countries_ISO = [country.alpha_3 for country in pycountry.countries]
unique_country_names = [country.name for country in pycountry.countries]
pattern = '|'.join(map(re.escape, unique_country_names))

unique_dict = {#mapping dictonary
    'hazards' : hazard_main_types_emdat_extended,
    'country' : unique_country_names,
    'startYear' : np.arange(1980, 2025).tolist(),
    'startMonth' : np.arange(1, 13).tolist(),
    'startDay' : np.arange(1, 32).tolist(),
    'endYear' : np.arange(1980, 2025).tolist(),
    'endMonth' : np.arange(1, 13).tolist(),
    'endDay' : np.arange(1, 32).tolist(),
    'impactSubtype' : impactSubtype_list
}

matching_cols = unique_dict.keys()
ext_vect_df = pd.DataFrame(columns=matching_cols)
lab_vect_df = pd.DataFrame(columns=matching_cols)


#vectorize
for col in matching_cols:
    ext_vect_df[col] = extracted_df[col].apply(vectorize, unique_values=unique_dict[col])
    lab_vect_df[col] = labelled_df[col].apply(vectorize, unique_values=unique_dict[col])

In [50]:
ext_vect_df

,hazards,country,startYear,startMonth,startDay,endYear,endMonth,endDay,impactSubtype
0,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, ..."
3,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
...,...,...,...,...,...,...,...,...,...
270,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
271,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
272,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
273,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0,

## Match

In [67]:
#compute cosine distance
from sklearn.metrics.pairwise import cosine_similarity

cos_dist_mat = np.full((len(ext_vect_df), len(lab_vect_df), len(matching_cols)), np.nan)
for k, col in enumerate(matching_cols):
    # Convert Series of arrays/lists to 2D numpy arrays
    X = np.stack(ext_vect_df[col].values) #nsamples, nfeatures
    Y = np.stack(lab_vect_df[col].values)
    # Compute cosine similarity
    sim_matrix = cosine_similarity(X, Y)
    cos_dist_mat[:,:,k] = cosine_similarity(X, Y)

In [74]:
np.unique(cos_dist_mat)

array([0.        , 0.5       , 0.70710678, 1.        , 1.        ])